# Rebuild the locked runtime offline

Restore a new runtime from the preserved archives. Default execution only validates the configuration and archives and describes the plan. Set TRACE_LAB_REBUILD=1 explicitly to execute installation into a new outputs/environments/ child. Existing prefixes and claimed attempts are never overwritten. Notebook tooling is not installed into the scientific runtime.


In [ ]:
from pathlib import Path
import os
_candidate = Path(os.environ.get('TRACE_LAB_ROOT', Path.cwd())).expanduser().resolve()
_candidates = [_candidate] if os.environ.get('TRACE_LAB_ROOT') else [_candidate, *_candidate.parents]
ROOT = next((p for p in _candidates if (p / '.trace-lab-root').is_file() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open inside trace-lab or set TRACE_LAB_ROOT to the clone')
# Run definition cells in this fresh kernel; this notebook has no execution side effects.
get_ipython().run_line_magic('run', '"' + str(ROOT / 'notebooks/library/configuration.ipynb') + '"')
ROOT = workspace_root(ROOT)


## Rebuild helpers and exclusive target reservation


In [ ]:
def plan_rebuild(root, assets, target):
    root = Path(root).resolve()
    destination = resolve_path(str(target), root)
    allowed = (root / 'outputs/environments').resolve()
    if allowed not in destination.parents:
        raise ConfigurationError('Rebuild target must be a new child of outputs/environments/')
    if destination.exists():
        raise FileExistsError('Refusing existing environment: ' + str(destination))
    conda = require_asset(assets, 'conda_executable')
    archive_root = require_asset(assets, 'runtime_archives')
    records = json.loads((root / 'environment/runtime/artifact_manifest.json').read_text())
    entries = []
    for row in records:
        path = (archive_root / row['path']).resolve()
        if archive_root not in path.parents:
            raise ConfigurationError('Archive path escapes configured root')
        if not path.is_file() or path.stat().st_size != row['bytes'] or file_sha256(path) != row['sha256']:
            raise ConfigurationError('Missing or changed archive: ' + str(path))
        if row['kind'] == 'conda':
            entries.append(path.as_uri() + '#' + row['md5'])
    provenance = json.loads((root / 'environment/runtime/provenance.json').read_text())
    for row in provenance['records']:
        if file_sha256(root / 'environment/runtime' / row['record']) != row['repository_sha256']:
            raise ConfigurationError('Preserved environment record changed: ' + row['record'])
    return {'target': str(destination), 'conda': str(conda), 'archive_root': str(archive_root),
            'verified_artifacts': len(records), 'verified_bytes': sum(row['bytes'] for row in records),
            'explicit_text': '@EXPLICIT\n' + '\n'.join(entries) + '\n'}

def execute_rebuild(root, plan):
    destination = Path(plan['target'])
    if destination.exists():
        raise FileExistsError('Refusing existing environment: ' + str(destination))
    # An exclusive sibling claim prevents simultaneous creation of the same target.
    destination.parent.mkdir(parents=True, exist_ok=True)
    claim = destination.parent / (destination.name + '.rebuild-claim')
    with claim.open('x') as stream:
        stream.write(datetime.now(timezone.utc).isoformat() + '\n')
    evidence = Path(root) / 'outputs/rebuilds' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])
    evidence.mkdir(parents=True, exist_ok=False)
    explicit = evidence / 'conda-explicit-local.txt'
    explicit.write_text(plan['explicit_text'])
    commands = [
        [plan['conda'], 'create', '--prefix', str(destination), '--file', str(explicit), '--offline', '--copy', '--no-default-packages', '--yes'],
        [str(destination / 'bin/python'), '-m', 'pip', '--isolated', 'install', '--no-index', '--find-links', str(Path(plan['archive_root']) / 'artifacts/wheels'), '--require-hashes', '--no-cache-dir', '-r', str(Path(root) / 'environment/runtime/requirements.lock.txt')],
    ]
    env = dict(os.environ, CONDA_PREFIX_DATA_INTEROPERABILITY='false', CONDA_REGISTER_ENVS='false', CONDA_AUTO_UPDATE_CONDA='false',
        CONDA_PKGS_DIRS=str(evidence / 'conda-cache'), PIP_DISABLE_PIP_VERSION_CHECK='1', PYTHONDONTWRITEBYTECODE='1')
    result = {'status': 'RUNNING', 'target': str(destination), 'verified_artifacts': plan['verified_artifacts'], 'commands': []}
    try:
        with (evidence / 'rebuild.log').open('w') as log:
            for command in commands:
                completed = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, env=env, timeout=execution_settings(root)['rebuild_timeout_seconds'])
                result['commands'].append({'argv': command, 'exit_code': completed.returncode})
                if completed.returncode:
                    raise RuntimeError('Rebuild command failed; see ' + str(evidence / 'rebuild.log'))
        result['runtime_comparison'] = compare_runtime(root, runtime_probe(destination / 'bin/python'))
        if result['runtime_comparison']['status'] != 'PASS':
            raise AssertionError(result['runtime_comparison'])
        result['status'] = 'PASS'
    except BaseException as exc:
        result['status'] = 'FAIL'; result['error'] = repr(exc)
        raise
    finally:
        (evidence / 'result.json').write_text(json.dumps(result, indent=2) + '\n')
    return result, evidence


## Validate prerequisites and prepare the plan


In [ ]:
assets = load_assets(ROOT)
execute = os.environ.get('TRACE_LAB_REBUILD', '0') == '1'
prerequisites = [row for row in asset_status(assets) if row['asset'] in ['runtime_archives','conda_executable'] and row['status'] != 'AVAILABLE']
plan = None
if prerequisites:
    print(json.dumps({'status': 'BLOCKED', 'missing': prerequisites}, indent=2))
    if execute:
        raise ConfigurationError('Rebuild requested without required archives/conda')
else:
    target = os.environ.get('TRACE_LAB_REBUILD_TARGET', 'outputs/environments/solid-runtime')
    plan = plan_rebuild(ROOT, assets, target)
    print(json.dumps({k:v for k,v in plan.items() if k != 'explicit_text'}, indent=2))
    print('Mode:', 'EXECUTE' if execute else 'PLAN_ONLY')


## Execute only when explicitly selected


In [ ]:
if execute and plan is not None:
    result, evidence = execute_rebuild(ROOT, plan)
    print(json.dumps(result, indent=2))
    print('Local evidence:', evidence)
else:
    print('No installation performed. Set TRACE_LAB_REBUILD=1 and use a new target to execute the plan.')
